In [1]:
from pathlib import Path
import json, hashlib, csv, collections, zipfile
ROOT=next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()); OUT=ROOT/'cleanup_audit_2026-11-13'
ARCHIVE=ROOT.parent/'Minor project archive'/'cleanup_2026-11-13'
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
files=sorted(p for p in ROOT.rglob('*') if p.is_file() and OUT not in p.parents)
texts={}; scopes={}; parse_errors=[]
for p in files:
 if '.notebook-runtime' not in p.parts and p.suffix.lower() in ['.ipynb','.py','.md','.txt','.json','.yaml','.yml','.toml','.ps1']:
  try:
   s=p.read_text(encoding='utf-8-sig')
   if p.suffix=='.ipynb':
    n=json.loads(s);s='\n'.join(''.join(c.get('source',[])) for c in n.get('cells',[]));scopes[p.relative_to(ROOT).as_posix()]=s[:700]
   texts[p.relative_to(ROOT).as_posix()]=s.replace('\\','/').lower()
  except Exception as e:parse_errors.append({'path':str(p),'error':str(e)})
print('Inventory',len(files),'files; reference sources',len(texts),flush=True)
rows=[];byhash=collections.defaultdict(list)
for num,p in enumerate(files):
 if num%250==0: print('Hash progress',num,'/',len(files),flush=True)
 rel=p.relative_to(ROOT).as_posix();digest=sha(p);byhash[digest].append(rel)
 refs=[k for k,s in texts.items() if k!=rel and (rel.lower() in s or p.name.lower() in s)] if '.notebook-runtime' not in p.parts else []
 rows.append({'filename':p.name,'path':rel,'absolute_path':str(p),'bytes':p.stat().st_size,'sha256':digest,'references':refs})
lookup={r['path']:r for r in rows}
# Explicit historical hash requirements, resolved against their documented waveform-project root.
wave=ROOT/'cnn/cnn wave'; baseline=[]
for name in ['eew_causal_v3/configs/protected_sources_before.json','eew_causal_v2/original_files_manifest_initial.json']:
 for rel,expected in json.loads((wave/name).read_text()).items():
  path=(wave/rel).relative_to(ROOT).as_posix();actual=lookup.get(path,{}).get('sha256')
  baseline.append({'manifest':name,'path':path,'expected':expected.lower(),'actual':actual,'match':actual==expected.lower()})
protected={b['path'] for b in baseline}
zip_evidence=[]
for p in files:
 if p.suffix!='.zip':continue
 members=[]
 with zipfile.ZipFile(p) as z:
  for item in z.infolist():
   if item.is_dir():continue
   digest=hashlib.sha256(z.read(item)).hexdigest()
   options=[p.parent/item.filename,p.with_suffix('')/item.filename]
   match=next((q for q in options if q.is_file() and lookup.get(q.relative_to(ROOT).as_posix(),{}).get('sha256')==digest),None)
   members.append({'member':item.filename,'sha256':digest,'retained_match':match.relative_to(ROOT).as_posix() if match else None})
 zip_evidence.append({'path':p.relative_to(ROOT).as_posix(),'members':members,'all_members_identical':all(m['retained_match'] for m in members)})
zipmap={z['path']:z for z in zip_evidence}
# Preserve any file registered by its current digest in an existing provenance JSON/Markdown.
for r in rows:
 r['hash_references']=[k for k,s in texts.items() if k!=r['path'] and r['sha256'] in s] if '.notebook-runtime/' not in r['path'] else []
 if r['hash_references']:protected.add(r['path'])
for r in rows:
 rel=r['path'];p=Path(rel);low=rel.lower(); ext=p.suffix.lower()
 r['duplicate_paths']=[x for x in byhash[r['sha256']] if x!=rel]
 r['protected']=rel in protected or '/eew_causal_v3/' in low
 r['classification']='UNKNOWN';r['reason']='Insufficient evidence of safe removal; retain in place.'
 if '/eew_causal_v3/' in low or rel=='dataset_2.csv' or ext in ['.h5','.hdf5','.npz','.npy','.csv','.pkl'] or any(x in low for x in ['scaler','normalization','split_manifest','arrival','metadata','hashes','provenance','manifest']):
  r.update(classification='KEEP_CORE',reason='Protected data, preprocessing, identities, provenance, or current v3 artifact. Legacy datasets retained but not approved for EEW training.')
 elif '/eew_causal_v2/' in low:
  r.update(classification='KEEP_REPRODUCIBILITY',reason='v3 simulator dynamically reads v2 generator definitions; v2 audit records original failure controls. Protected baseline and replay dependencies prevent removal.')
 elif ext in ['.ipynb','.py','.pt','.pth','.png','.svg','.jpg','.json','.md','.txt','.db']:
  r.update(classification='KEEP_REPRODUCIBILITY',reason='Experiment source, checkpoint, performance evidence, configuration, or runtime state. Removal could break reproduction; no complete validated replacement established.')
 if r['protected'] and r['classification']=='UNKNOWN':r.update(classification='KEEP_REPRODUCIBILITY',reason='Registered historical integrity dependency; keep original location and bytes.')
 if rel in zipmap:
  z=zipmap[rel]
  if z['all_members_identical'] and not r['references'] and not r['protected']:
   r.update(classification='ARCHIVE_CANDIDATE',reason='Every ZIP member exactly matches retained extracted content; no literal source references, no registered protection, no unique experiment evidence. Packaging copy unnecessary for training or reproduction.')
  else:r.update(classification='KEEP_REPRODUCIBILITY',reason='ZIP contains unmatched content, recorded protection, or unresolved references; not safe to archive as a redundant export.')
 if '.notebook-runtime/' in rel:r.update(classification='KEEP_REPRODUCIBILITY',reason='Notebook execution runtime; preserve environment, excluded from project dependency source scan.')
 r['action']='RETAIN'
candidates=[r for r in rows if r['classification']=='ARCHIVE_CANDIDATE']
plan=[{'source':r['absolute_path'],'destination':str(ARCHIVE/r['path']),'bytes':r['bytes'],'sha256':r['sha256'],'reason':r['reason'],'rollback':'Move destination back to source only if source is absent and destination SHA256 matches.'} for r in candidates]
summary={k:{'files':sum(r['classification']==k for r in rows),'bytes':sum(r['bytes'] for r in rows if r['classification']==k)} for k in ['KEEP_CORE','KEEP_REPRODUCIBILITY','ARCHIVE_CANDIDATE','UNKNOWN']}
audit={'date':'2026-11-13','project':str(ROOT),'archive':str(ARCHIVE),'total_files':len(rows),'total_bytes':sum(r['bytes'] for r in rows),'summary':summary,'protection_checks':baseline,'parse_errors':parse_errors,'zip_evidence':zip_evidence,'archive_plan':plan,'actually_archived':[],'dependency_limit':'Static notebook-source/script/document basename and full-path references plus registered content hashes; dynamic paths cannot be exhaustively resolved. Uncertain files retained. Notebook outputs not treated as executable dependencies.','scope_findings':{'catalog_CNN':'cnn/cnn wave/phase1_cnn_forecasting.ipynb explicitly targets future recorded seismicity, not same-event EEW; retained with results and scaler/splits.','catalog_BiLSTM':'bilstm new updated/phase2_bilstm_training.ipynb and phase3_modern_catalog_analysis.ipynb explicitly target next retained ML catalog event; retained for historical reproducibility, not approved EEW inputs.','legacy_seq20':'bilstm/ provenance reports leave timing and feature construction unresolved; retain UNKNOWN scientific provenance, no automatic EEW reuse.','failed_waveforms':'v2 and original Phase11 evidence retained; v3 directly depends on original generator and metadata paths. No failed-data repair or relabeling as valid.','hybrid':'Legacy hybrid sources/checkpoints retained. Architecture name alone does not establish causal feature availability or EEW target comparability.'}}
(OUT/'cleanup_manifest.json').write_text(json.dumps({'files':rows,'audit':audit},indent=2),encoding='utf-8')
with (OUT/'cleanup_manifest.csv').open('w',newline='',encoding='utf-8') as f:
 fields=['filename','path','bytes','sha256','classification','protected','reason','action','references','hash_references','duplicate_paths'];w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader()
 for r in rows:w.writerow({k:json.dumps(r[k]) if isinstance(r[k],list) else r[k] for k in fields})
(OUT/'archive_plan.json').write_text(json.dumps(plan,indent=2),encoding='utf-8')
(OUT/'rollback_manifest.json').write_text(json.dumps({'status':'PLANNED_NOT_MOVED','moves':plan},indent=2),encoding='utf-8')
lines=['# Strict EEW cleanup audit — 2026-11-13','','Dataset verification completed separately: v3 HDF5 checks PASS; overall approval remains CONDITIONAL because no v3 NPZ exists. No training started.','','## Inventory','',f"Inventoried and SHA256-hashed {len(rows)} existing files, {audit['total_bytes']:,} bytes ({audit['total_bytes']/1024**3:.3f} GiB). Per-file classifications, sizes, hashes, references and duplicate matches are in cleanup_manifest.csv/json.",'','| Classification | Files | Bytes |','|---|---:|---:|']
for k,v in summary.items():lines.append(f"| {k} | {v['files']} | {v['bytes']:,} |")
lines+=['','## Retained','', 'All original catalogs, CSVs, HDF5/NPZ/NPY data, manifests, scalers, arrival metadata and current v3 artifacts remain in place. Original waveform CNN notebook, best/last checkpoints, configuration, history, performance curves and source datasets are retained. Forecasting notebooks/checkpoints remain historical evidence, not authorized EEW training inputs.','', 'V3 loads the v2 generator definitions directly. Archiving v2 would break active paths. Failed v2 and Phase11 datasets/reports retain scientific evidence; newer v3 does not waive old integrity requirements. Duplicate notebooks/plots are listed by exact hash but retained where paths, provenance or reproducibility remain relevant. The hybrid ZIP has one differing report and is retained.','','## Proposed archive','']
for p in plan:lines += [f"- `{Path(p['source']).relative_to(ROOT).as_posix()}` — {p['bytes']:,} bytes; all 41 members exactly match the retained extracted result directory; no source reference or integrity constraint found. Destination: `{p['destination']}`."]
lines+=['','## Actual archive and integrity','','Archive not yet executed; planned bytes: '+str(sum(x['bytes'] for x in plan))+'. External destination requires filesystem permission. Rollback paths and hashes are recorded before any move.','',f"Historical protected-baseline checks: {sum(b['match'] for b in baseline)}/{len(baseline)} matching entries ({len(set(b['path'] for b in baseline))} unique paths). Existing integrity-controlled memory/status files were deliberately not updated. Whole-inventory before/after preservation will be checked after archival.",'','## Unresolved dependencies and future work','','Static references cannot prove absence of every dynamic dependency; uncertain files stay. Legacy seq20 input timing/provenance remains unresolved. Tabular BiLSTM must use information available by the same observation cutoff and comparable same-event targets/splits as CNN; historical next-event models are not interchangeable. S-arrival masks and oracle alignment are evaluation metadata, not operational trigger capabilities.','', 'Future outputs: notebook-only executable ML code, reuse verified datasets, retain essential checkpoints and curves, concise validation reports, no redundant large datasets. No training until dataset approval; no permanent deletion.']
(OUT/'cleanup_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print(json.dumps({'summary':summary,'historical_baseline_matches':sum(b['match'] for b in baseline),'baseline_entries':len(baseline),'archive_plan':plan,'parse_errors':parse_errors},indent=2))


Inventory 24412 files; reference sources 964
Hash progress 0 / 24412
Hash progress 250 / 24412
Hash progress 500 / 24412
Hash progress 750 / 24412
Hash progress 1000 / 24412
Hash progress 1250 / 24412
Hash progress 1500 / 24412
Hash progress 1750 / 24412
Hash progress 2000 / 24412
Hash progress 2250 / 24412
Hash progress 2500 / 24412
Hash progress 2750 / 24412
Hash progress 3000 / 24412
Hash progress 3250 / 24412
Hash progress 3500 / 24412
Hash progress 3750 / 24412
Hash progress 4000 / 24412
Hash progress 4250 / 24412
Hash progress 4500 / 24412
Hash progress 4750 / 24412
Hash progress 5000 / 24412
Hash progress 5250 / 24412
Hash progress 5500 / 24412
Hash progress 5750 / 24412
Hash progress 6000 / 24412
Hash progress 6250 / 24412
Hash progress 6500 / 24412
Hash progress 6750 / 24412
Hash progress 7000 / 24412
Hash progress 7250 / 24412
Hash progress 7500 / 24412
Hash progress 7750 / 24412
Hash progress 8000 / 24412
Hash progress 8250 / 24412
Hash progress 8500 / 24412
Hash progress 87

In [2]:
# Final preservation check after user-confirmed deletion of the redundant ZIP.
# This cell does not move, delete, restore, or modify any original artifact.
from pathlib import Path
import json,hashlib,csv
from concurrent.futures import ThreadPoolExecutor
ROOT=next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file());OUT=ROOT/'cleanup_audit_2026-11-13'
m=json.loads((OUT/'cleanup_manifest.json').read_text());rows=m['files'];audit=m['audit']
removed='bilstm new updated/phase3_bilstm_results.zip'
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
def verify(r):
 p=Path(r['absolute_path'])
 try:return {'path':r['path'],'match':p.is_file() and p.stat().st_size==r['bytes'] and sha(p)==r['sha256']}
 except Exception as e:return {'path':r['path'],'match':False,'error':str(e)}
retained=[r for r in rows if r['path']!=removed]
with ThreadPoolExecutor(max_workers=16) as pool:checks=list(pool.map(verify,retained))
failed=[x for x in checks if not x['match']]
assert not (ROOT/removed).exists(),'User-deleted file unexpectedly restored; review required'
deleted=next(r for r in rows if r['path']==removed);deleted['action']='USER_DELETED'
deleted['action_evidence']='User explicitly confirmed: i deleted no need of it. Agent move failed at pre-move source hash check; no archive or deletion executed by agent.'
audit['actually_archived']=[];audit['user_deleted']=[{'path':removed,'bytes':deleted['bytes'],'sha256_before':deleted['sha256'],'evidence':deleted['action_evidence']}]
audit['post_cleanup_integrity']={'status':'PASS' if not failed else 'FAIL','retained_files_hashed':len(checks),'failures':failed,'archived_files':0,'archived_bytes':0,'user_deleted_files':1,'user_deleted_bytes':deleted['bytes'],'retained_bytes':sum(r['bytes'] for r in retained)}
(OUT/'cleanup_manifest.json').write_text(json.dumps(m,indent=2),encoding='utf-8')
with (OUT/'cleanup_manifest.csv').open('w',newline='',encoding='utf-8') as f:
 fields=['filename','path','bytes','sha256','classification','protected','reason','action','references','hash_references','duplicate_paths'];w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader()
 for r in rows:w.writerow({k:json.dumps(r[k]) if isinstance(r[k],list) else r[k] for k in fields})
(OUT/'rollback_manifest.json').write_text(json.dumps({'status':'NO_AGENT_MOVES_USER_DELETED_CANDIDATE','moves':[],'original_plan':audit['archive_plan'],'user_deleted':audit['user_deleted'],'rollback':'No archive copy exists, so rollback of the deleted ZIP is unavailable. Its 41 extracted members were verified identical before deletion and remain retained. Do not recreate the ZIP unless requested.','integrity':audit['post_cleanup_integrity']},indent=2),encoding='utf-8')
p=OUT/'cleanup_report.md';s=p.read_text()
s=s.replace('Archive not yet executed; planned bytes: '+str(deleted['bytes'])+'. External destination requires filesystem permission. Rollback paths and hashes are recorded before any move.',f"Actually archived by agent: **0 files, 0 bytes**. The user deleted the proposed ZIP during the audit and explicitly confirmed that it was no longer needed. The move command stopped at its source-existence/hash check before any move or archive directory creation. User-deleted size: **{deleted['bytes']:,} bytes ({deleted['bytes']/1024**2:.2f} MiB)**. This is not an archive operation, and no archive rollback copy exists. All 41 extracted ZIP members remain in the project and were included in post-cleanup hashing. The original path, SHA256 and proposed destination remain documented.")
s=s.replace('Whole-inventory before/after preservation will be checked after archival.',f"Post-cleanup SHA256 verification: **{len(checks)-len(failed)}/{len(checks)} retained files unchanged**, excluding only the explicitly user-deleted ZIP. Integrity status: **{audit['post_cleanup_integrity']['status']}**. All protected originals and current v3 files remain retained.")
s+='\nUnknown items were retained: mostly training-runtime compiled/cache files, the compressed source snapshot, a temporary-looking CSV and a log. Names alone do not establish dispensability. No newer replacement was assumed. No source data or experiment artifact was deleted by the agent.\n'
p.write_text(s,encoding='utf-8')
print(json.dumps(audit['post_cleanup_integrity'],indent=2))


{
  "status": "PASS",
  "retained_files_hashed": 24411,
  "failures": [],
  "archived_files": 0,
  "archived_bytes": 0,
  "user_deleted_files": 1,
  "user_deleted_bytes": 16322280,
  "retained_bytes": 4771619138
}
